# Memory Optimization

**Module:** 13 — AI Memory

Latency, cost, compression, caching, and quality trade-offs.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Identify optimization levers: recall size, embedding model, caching, compression
- Implement summarization and packing strategies
- Apply a latency playbook for memory-augmented calls
- Measure quality vs cost trade-offs


## Optimization Levers

### Definition
Knobs that change memory system latency, cost, and answer quality — often in tension.

### Why it matters
Memory can dominate p95 if you embed synchronously, fetch large top-k, or inject huge prompts.

### How it works
Tune: embedding batching, ANN params (`ef`, probes), top-k, reranker cost, cache hits, prompt budgets, async writeback.

### Intuition
Every millisecond on the retrieve path is paid on every user turn.

### Pitfalls
- Optimizing only average latency while p95 burns
- Cutting top-k so hard that recall collapses
- Caching across tenants

### When to use
Any production traffic; start measuring before clever tricks.


| Lever | Speeds up | Risk if overused |
|-------|-----------|------------------|
| Lower top-k | Retrieve + prompt | Miss relevant memory |
| Smaller embed model | Index + query embed | Weaker recall |
| Skip reranker | Latency/cost | Noisy inject |
| Cache embeddings | Repeat queries | Stale vectors |
| Async writeback | User-perceived latency | Lost writes on crash |
| Summarize history | Prompt tokens | Detail loss |

```mermaid
flowchart LR
  Q[Query] --> C{Embed cache?}
  C -->|hit| S[Search]
  C -->|miss| E[Embed] --> S
  S --> R[Rerank optional]
  R --> P[Pack]
  P --> LLM
```


In [ ]:
# Demo 1: embedding cache
from functools import lru_cache
import hashlib

def fake_embed(text: str) -> list[float]:
    h = hashlib.sha256(text.encode()).digest()
    return [b / 255.0 for b in h[:8]]

@lru_cache(maxsize=256)
def cached_embed(text: str) -> tuple:
    return tuple(fake_embed(text))

for _ in range(3):
    v = cached_embed("User prefers UTC")
print("cache info", cached_embed.cache_info())
print("dim", len(v))


In [ ]:
# Demo 2: cost model for memory-augmented turns
def turn_cost_usd(prompt_tokens, completion_tokens, embed_tokens, prices=None):
    prices = prices or {
        "prompt_per_1k": 0.00015,
        "completion_per_1k": 0.0006,
        "embed_per_1k": 0.00002,
    }
    return (
        prompt_tokens / 1000 * prices["prompt_per_1k"]
        + completion_tokens / 1000 * prices["completion_per_1k"]
        + embed_tokens / 1000 * prices["embed_per_1k"]
    )

baseline = turn_cost_usd(2000, 300, 200)
optimized = turn_cost_usd(900, 300, 0)  # smaller prompt, cached embed
print({"baseline": baseline, "optimized": optimized, "savings_pct": round(100 * (1 - optimized / baseline), 1)})


### Try it yourself — Levers

1. Add a TTL to the embed cache (store timestamp; invalidate after N seconds).
2. Plot (print table) cost vs injected_chars for injected_chars in [0,200,400,800].


## Summarization & Compression

### Definition
Techniques that reduce tokens while preserving decision-critical content: hierarchical summaries, entity facts extraction, and lossless compression of structure.

### Why it matters
Context is expensive; compression is how long agents survive.

### How it works
Maintain rolling summaries; extract structured facts; keep raw episodes cold. Prefer extractive constraints for safety-critical fields.

### Intuition
Bullet diary > novel. Structured JSON > poetry for machines.

### Pitfalls
- Abstractive summaries inventing constraints
- Compressing away 'never do X' instructions
- Single giant summary that cannot be queried

### When to use
Long threads, multi-day agents, high-throughput bots.


In [ ]:
# Demo 3: extractive compression — keep constraint sentences
CONSTRAINT_MARKERS = ("always", "never", "must", "prefer", "deadline")

def compress_messages(messages: list[str], max_lines=5) -> list[str]:
    constraints = [m for m in messages if any(k in m.lower() for k in CONSTRAINT_MARKERS)]
    others = [m for m in messages if m not in constraints]
    # constraints first, then recent others
    picked = constraints + others[-max_lines:]
    return picked[:max_lines]

msgs = [
    "hello",
    "We must never email PII",
    "lol",
    "Deploy failed",
    "User prefers UTC",
    "retrying",
    "success",
]
print(compress_messages(msgs, max_lines=4))


In [ ]:
# Demo 4: hierarchical session summary structure
session = {
    "rolling_summary": "User is configuring Phoenix billing exports.",
    "entities": {"project": "Phoenix", "pref_timezone": "UTC"},
    "recent_turns": ["asked about CSV columns", "confirmed invoice_id required"],
}

def pack_session(session, budget=160):
    parts = [
        f"SUMMARY: {session['rolling_summary']}",
        "ENTITIES: " + ", ".join(f"{k}={v}" for k, v in session["entities"].items()),
        "RECENT: " + " | ".join(session["recent_turns"]),
    ]
    out = []
    used = 0
    for p in parts:
        if used + len(p) > budget:
            break
        out.append(p)
        used += len(p)
    return "\n".join(out)

print(pack_session(session))


## Latency Playbook

### Definition
A prioritized checklist to reduce p95 for memory-augmented generation.

### Why it matters
Users feel memory latency as 'the bot is slow,' even when the LLM is fine.

### How it works
1. Cache embeddings for repeated queries  
2. Parallelize retrieve with other prep work  
3. Use filters to shrink ANN candidate sets  
4. Make writeback asynchronous with a durable queue  
5. Prefetch memories on session start  
6. Prefer local/lightweight rerankers for top 20 only

### Intuition
Move memory work off the critical path whenever correctness allows.

### Pitfalls
- Async writes without retry/idempotency
- Prefetch that fetches the wrong tenant

### When to use
Interactive chat/voice agents (see also Module 16 latency budgets).


In [ ]:
# Demo 5: parallel retrieve simulation
import time
from concurrent.futures import ThreadPoolExecutor

def embed_ms():
    time.sleep(0.05)
    return [0.1, 0.2]

def search_ms(vec):
    time.sleep(0.05)
    return ["mem-a", "mem-b"]

def sequential():
    t0 = time.time()
    # pretend other work
    time.sleep(0.05)
    v = embed_ms()
    hits = search_ms(v)
    return time.time() - t0, hits

def parallel_other_work():
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut = ex.submit(lambda: search_ms(embed_ms()))
        time.sleep(0.05)  # other work overlaps
        hits = fut.result()
    return time.time() - t0, hits

print("sequential", round(sequential()[0], 3))
print("overlapped", round(parallel_other_work()[0], 3))


### Try it yourself — Optimize a path

1. Given top_k=20, rerank_model=expensive, propose a 3-step plan to cut latency 40% without killing recall.
2. Implement async writeback mock: `enqueue(record)` + `flush()` processing queue.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `p95` | 95th percentile latency |
| `writeback` | Persisting memory after response path |
| `rolling summary` | Continuously updated compressed history |
| `embed cache` | Memoization of embedding API results |


## Quality–Cost Pareto Thinking

```
High quality ↑
  |   A (big top-k + rerank)
  |        B (cached embed + top-5 + light rerank)  ← sweet spot often
  |   C (no memory)
  +----------------→ Low cost
```

Measure with a frozen eval set whenever you turn a lever.


In [ ]:
# Pareto candidates
candidates = [
    {"name": "A", "recall": 0.82, "latency_ms": 220, "usd": 0.004},
    {"name": "B", "recall": 0.78, "latency_ms": 110, "usd": 0.002},
    {"name": "C", "recall": 0.55, "latency_ms": 20, "usd": 0.0005},
]

def dominated(a, b):
    # b dominates a if better/equal on all and better on some (higher recall, lower lat/cost)
    return (
        b["recall"] >= a["recall"] and b["latency_ms"] <= a["latency_ms"] and b["usd"] <= a["usd"]
        and (b["recall"] > a["recall"] or b["latency_ms"] < a["latency_ms"] or b["usd"] < a["usd"])
    )

frontier = [c for c in candidates if not any(dominated(c, o) for o in candidates if o is not c)]
print("frontier", [c["name"] for c in frontier])


In [ ]:
# Prompt budget allocator across memory types
def allocate(budget_tokens, weights):
    total_w = sum(weights.values())
    return {k: int(budget_tokens * w / total_w) for k, w in weights.items()}

print(allocate(800, {"procedure": 3, "semantic": 2, "episodic": 1, "stm": 2}))


### Try it yourself — Optimization deepen

1. Add a constraint: recall must be >= 0.75 when selecting from candidates.
2. Benchmark (print) packer time for 1000 fake memories.


## Key Takeaways

- Measure retrieve+pack+LLM as separate spans
- Compression must preserve constraints
- Cache carefully with tenant keys and TTLs
- Async writeback improves UX if made durable
